In [ ]:

!pip install pathway bokeh --quiet

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime
from datetime import datetime
import pathway as pw
import bokeh.plotting
import panel as pn

In [ ]:

github_url = "https://raw.githubusercontent.com/Krishtiy/final-project-data-anlytics/refs/heads/main/dataset%20(6).csv"
df = pd.read_csv(github_url)

In [ ]:

grouped_datasets = {group: data for group, data in df.groupby('SystemCodeNumber')}

In [ ]:

df['SystemCodeNumber'].unique()

In [ ]:

#all 14 lots together
df = df
print(df['SystemCodeNumber'].unique())
print(df.shape)

In [ ]:

df['TrafficConditionNearby'].unique()

In [ ]:

df['Timestamp'] = pd.to_datetime(
    df['LastUpdatedDate'] + ' ' + df['LastUpdatedTime'],
    format='%d-%m-%Y %H:%M:%S'
)
df = df.sort_values('Timestamp').reset_index(drop=True)
df["Timestamp"] = pd.to_datetime(df["Timestamp"])

In [ ]:
df.columns = df.columns.str.strip()

df_clean = df[[
    "Timestamp", "SystemCodeNumber", "Occupancy", "Capacity",
    "QueueLength", "TrafficConditionNearby", "IsSpecialDay",
    "VehicleType", "Latitude", "Longitude"
]]

df_clean.to_csv("parking_stream.csv", index=False)
print(f"Saved {len(df_clean)} rows across {df_clean['SystemCodeNumber'].nunique()} lots")

In [ ]:

class ParkingSchema(pw.Schema):
    Timestamp: str
    SystemCodeNumber: str
    Occupancy: int
    Capacity: int
    QueueLength: int
    TrafficConditionNearby: str
    IsSpecialDay: int
    VehicleType: str
    Latitude: float
    Longitude: float

In [ ]:

data = pw.demo.replay_csv("parking_stream.csv", schema=ParkingSchema, input_rate=100)

In [ ]:

fmt = "%Y-%m-%d %H:%M:%S"

data_with_time = data.with_columns(
    t=data.Timestamp.dt.strptime(fmt),
    day=data.Timestamp.dt.strptime(fmt).dt.strftime("%Y-%m-%dT00:00:00")
)

In [ ]:


import datetime

base_price = 10
alpha = 5

delta_window1 = (
    data_with_time.windowby(
        pw.this.t,
        instance=pw.this.day,
        window=pw.temporal.tumbling(datetime.timedelta(days=1)),
        behavior=pw.temporal.exactly_once_behavior()
    )
    .reduce(
        t=pw.this._pw_window_end,
        occ_max=pw.reducers.max(pw.this.Occupancy),
        occ_min=pw.reducers.min(pw.this.Occupancy),
        cap=pw.reducers.max(pw.this.Capacity),
    )
    .with_columns(

        price=base_price + alpha * (pw.this.occ_max - pw.this.occ_min) / pw.this.cap,
    )
)

In [ ]:

base_price = 10
alpha, beta, delta_coef = 0.5, 0.3, 0.4
lambda_ = 0.8


def get_traffic_weight(t: str) -> float:
    return float({"low": 0.6, "average": 1.0, "high": 1.4}.get(t, 1.0))

def get_vehicle_weight(v: str) -> float:
    return float({"car": 1.0, "bike": 0.6, "truck": 1.4, "cycle": 0.5}.get(v, 1.0))

def compute_price(
    occ_max: int,
    occ_min: int,
    cap: int,
    queue: int,
    special: int,
    vehicle: float,
    traffic: float
) -> float:
    return float(base_price) * (
        1.0 + lambda_ * (
            alpha * float(occ_max - occ_min) / float(cap)
            + beta * float(queue)
            + delta_coef * float(special)
            + vehicle
            + traffic
        ) / 5.0
    )

delta_window2 = (
    data_with_time.windowby(
        pw.this.t,
        instance=pw.this.day,
        window=pw.temporal.tumbling(datetime.timedelta(days=1)),
        behavior=pw.temporal.exactly_once_behavior()
    )
    .reduce(
        t=pw.this._pw_window_end,
        occ_max=pw.reducers.max(pw.this.Occupancy),
        occ_min=pw.reducers.min(pw.this.Occupancy),
        cap=pw.reducers.max(pw.this.Capacity),
        IsSpecialDay=pw.reducers.max(pw.this.IsSpecialDay),
        queue=pw.reducers.max(pw.this.QueueLength),
        traffic_str=pw.reducers.any(pw.this.TrafficConditionNearby),
        vehicle_str=pw.reducers.any(pw.this.VehicleType),
    )
    .with_columns(

        traffic=pw.apply(get_traffic_weight, pw.this.traffic_str),
        vehicle_weight=pw.apply(get_vehicle_weight, pw.this.vehicle_str),
    )
    .with_columns(
        price=pw.apply(
            compute_price,
            pw.this.occ_max,
            pw.this.occ_min,
            pw.this.cap,
            pw.this.queue,
            pw.this.IsSpecialDay,
            pw.this.vehicle_weight,
            pw.this.traffic,
        )
    )
)

In [ ]:
base_price = 10
surge_factor = 1.5
discount_factor = 0.8
threshold_high = 0.8
threshold_low = 0.3

delta_window3 = (
    data_with_time.windowby(
        pw.this.t,
        instance=pw.this.day,
        window=pw.temporal.tumbling(datetime.timedelta(days=1)),
        behavior=pw.temporal.exactly_once_behavior()
    )
    .reduce(
        t=pw.this._pw_window_end,
        occ_max=pw.reducers.max(pw.this.Occupancy),
        occ_min=pw.reducers.min(pw.this.Occupancy),
        cap=pw.reducers.max(pw.this.Capacity),
        IsSpecialDay=pw.reducers.max(pw.this.IsSpecialDay),
        queue=pw.reducers.max(pw.this.QueueLength),
        traffic_str=pw.reducers.any(pw.this.TrafficConditionNearby),
        vehicle_str=pw.reducers.any(pw.this.VehicleType),
    )
    .with_columns(
        occupancy_rate=pw.this.occ_max / pw.this.cap,
        traffic=pw.apply(
            lambda t: {"low": 0.6, "average": 1.0, "high": 1.4}.get(t, 1.0),
            pw.this.traffic_str
        ),
        vehicle_weight=pw.apply(
            lambda v: {"car": 1.0, "bike": 0.6, "truck": 1.4, "cycle": 0.5}.get(v, 1.0),
            pw.this.vehicle_str
        ),
    )
    .with_columns(
        price_multiplier=pw.apply(
            lambda rate: surge_factor if rate > threshold_high
                         else (discount_factor if rate < threshold_low else 1.0),
            pw.this.occupancy_rate
        )
    )
    .with_columns(
        price=pw.apply(
            lambda base, multiplier, special, traffic, vehicle, queue: max(5.0, min(
                base * multiplier
                * (1.2 if special else 1.0)
                * traffic
                * vehicle
                * (1 + 0.05 * queue),
                20.0
            )),
            base_price,
            pw.this.price_multiplier,
            pw.this.IsSpecialDay,
            pw.this.traffic,
            pw.this.vehicle_weight,
            pw.this.queue,
        )
    )
)

In [ ]:
import math

lot_daily = (
    data_with_time.windowby(
        pw.this.t,
        instance=pw.this.SystemCodeNumber,
        window=pw.temporal.tumbling(datetime.timedelta(days=1)),
        behavior=pw.temporal.exactly_once_behavior()
    )
    .reduce(
        t=pw.this._pw_window_end,
        lot=pw.reducers.any(pw.this.SystemCodeNumber),
        lat=pw.reducers.any(pw.this.Latitude),
        lon=pw.reducers.any(pw.this.Longitude),
        occ_max=pw.reducers.max(pw.this.Occupancy),
        occ_min=pw.reducers.min(pw.this.Occupancy),
        cap=pw.reducers.max(pw.this.Capacity),
        queue=pw.reducers.max(pw.this.QueueLength),
        IsSpecialDay=pw.reducers.max(pw.this.IsSpecialDay),
        traffic_str=pw.reducers.any(pw.this.TrafficConditionNearby),
        vehicle_str=pw.reducers.any(pw.this.VehicleType),
    )
    .with_columns(
        occupancy_rate=pw.this.occ_max / pw.this.cap,
        traffic=pw.apply(get_traffic_weight, pw.this.traffic_str),
        vehicle_weight=pw.apply(get_vehicle_weight, pw.this.vehicle_str),
    )
    .with_columns(
        base_demand_price=pw.apply(
            compute_price,
            pw.this.occ_max, pw.this.occ_min, pw.this.cap,
            pw.this.queue, pw.this.IsSpecialDay,
            pw.this.vehicle_weight, pw.this.traffic,
        )
    )
)

def competitive_price(base_price: float, occupancy_rate: float, lat: float, lon: float) -> float:
    if occupancy_rate > 0.85:
        multiplier = 1.1
    elif occupancy_rate < 0.3:
        multiplier = 0.85
    else:
        multiplier = 1.0
    return float(max(5.0, min(base_price * multiplier, 20.0)))

def reroute_flag(occupancy_rate: float) -> str:
    if occupancy_rate > 0.9:
        return "REROUTE_SUGGESTED"
    elif occupancy_rate > 0.75:
        return "HIGH_DEMAND"
    else:
        return "NORMAL"

delta_competitive = (
    lot_daily
    .with_columns(
        price=pw.apply(
            competitive_price,
            pw.this.base_demand_price,
            pw.this.occupancy_rate,
            pw.this.lat,
            pw.this.lon,
        ),
        reroute_status=pw.apply(reroute_flag, pw.this.occupancy_rate),
    )
)

print("delta_competitive defined successfully")

In [ ]:

pn.extension()

pw.io.csv.write(delta_window1.select(pw.this.t, pw.this.price), "out_model1.csv")
pw.io.csv.write(delta_window2.select(pw.this.t, pw.this.price), "out_model2.csv")
pw.io.csv.write(delta_window3.select(pw.this.t, pw.this.price), "out_model3.csv")
pw.io.csv.write(
    delta_competitive.select(
        pw.this.t, pw.this.lot, pw.this.price,
        pw.this.occupancy_rate, pw.this.reroute_status
    ),
    "out_competitive.csv"
)

In [ ]:
# Write rerouting alerts to a CSV for review
pw.io.csv.write(
    delta_competitive.filter(pw.this.reroute_status != "NORMAL")
    .select(
        pw.this.t,
        pw.this.lot,
        pw.this.occupancy_rate,
        pw.this.price,
        pw.this.reroute_status,
    ),
    "rerouting_alerts.csv"
)

print("Rerouting alerts will be written to rerouting_alerts.csv during pw.run()")

In [ ]:
pw.io.csv.write(
    delta_competitive
    .filter(pw.this.reroute_status != "NORMAL")
    .select(pw.this.t, pw.this.lot, pw.this.occupancy_rate,
            pw.this.price, pw.this.reroute_status),
    "rerouting_alerts.csv"
)
print("Rerouting alerts will be written to rerouting_alerts.csv")

In [ ]:

pw.run()

In [ ]:
import os

files = ["out_model1.csv", "out_model2.csv", "out_model3.csv", "out_competitive.csv"]
for f in files:
    if os.path.exists(f):
        df_check = pd.read_csv(f)
        print(f"✓ {f} — {len(df_check)} rows")
        print(df_check.head(3))
        print()
    else:
        print(f"✗ {f} — NOT FOUND")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

fig, axes = plt.subplots(4, 1, figsize=(14, 20))
titles = [
    ("out_model1.csv", "Model 1 — Baseline Linear Price", "steelblue"),
    ("out_model2.csv", "Model 2 — Demand-Based Price", "darkgreen"),
    ("out_model3.csv", "Model 3 — Surge / Discount Price", "purple"),
    ("out_competitive.csv", "Model 4 — Competitive Pricing", "firebrick"),
]

for ax, (fname, title, color) in zip(axes, titles):
    df_out = pd.read_csv(fname)
    df_out['t'] = pd.to_datetime(df_out['t'])
    df_out = df_out.sort_values('t')

    if 'lot' in df_out.columns:
        for lot, grp in df_out.groupby('lot'):
            ax.plot(grp['t'], grp['price'], marker='o', markersize=2,
                    linewidth=1, label=lot, alpha=0.75)
        if 'reroute_status' in df_out.columns:
            reroute = df_out[df_out['reroute_status'] == 'REROUTE_SUGGESTED']
            if len(reroute) > 0:
                ax.scatter(reroute['t'], reroute['price'],
                           color='red', s=60, zorder=5,
                           label='Reroute suggested', marker='X')
        ax.legend(fontsize=6, ncol=4, loc='upper left', framealpha=0.7)
    else:
        ax.plot(df_out['t'], df_out['price'], color=color, marker='o', markersize=4)

    ax.set_title(title, fontsize=13)
    ax.set_xlabel("Date")
    ax.set_ylabel("Price ($)")
    ax.grid(True, alpha=0.3)

plt.tight_layout(pad=3)
plt.savefig("pricing_dashboard.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved to pricing_dashboard.png")